# Pendulum First — MuJoCo + PPO on Google Colab

**Begin here before training Mini Pupper.** Four observations, one continuous force action, one PPO model. Runs on CPU; GPU not required.

This notebook is a separate educational environment, not the Mini Pupper GPU training entry point.

## 1. Install dependencies
Run this cell on an ordinary Colab Python runtime.

In [ ]:
%pip -q install "mujoco>=3.3,<4" "gymnasium>=1,<2" "stable-baselines3>=2.5,<3"

## 2. Create a MuJoCo XML model
The cart slides left and right; the pole pivots on a hinge. Force is the sole action.

In [ ]:
%%writefile pendulum.xml
<mujoco model="cart_pole_standup">
  <compiler angle="radian"/>
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4"/>
  <visual><global offwidth="900" offheight="650"/></visual>
  <worldbody>
    <light pos="0 -3 4" dir="0 0 -1"/>
    <geom name="floor" type="plane" pos="0 0 -0.22" size="4 2 0.1" rgba="0.88 0.91 0.96 1"/>
    <body name="cart" pos="0 0 0">
      <joint name="cart_slide" type="slide" axis="1 0 0" limited="true" range="-2.5 2.5" damping="0.2"/>
      <geom name="cart_box" type="box" size="0.18 0.12 0.10" mass="1.0" rgba="0.15 0.43 0.78 1"/>
      <body name="pole" pos="0 0 0.10">
        <joint name="pole_hinge" type="hinge" axis="0 1 0" damping="0.002"/>
        <geom name="pole_rod" type="capsule" fromto="0 0 0 0 0 0.65" size="0.035" mass="0.20" rgba="0.97 0.64 0.16 1"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="cart_force" joint="cart_slide" gear="1" ctrllimited="true" ctrlrange="-20 20"/></actuator>
</mujoco>


## 3. Create a Gymnasium wrapper
Observation: `[x, angle, x_dot, angular_velocity]` (4); action: force scaled to ±20 N.

In [ ]:
%%writefile pendulum_env.py
"""Small Gymnasium environment: 4 observations, 1 continuous force action.

This is an educational custom environment, not the built-in Gymnasium one.
"""
from pathlib import Path
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np


class PendulumEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_path(str(Path(__file__).with_name('pendulum.xml')))
        self.data = mujoco.MjData(self.model)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32)
        self._steps = 0

    def _obs(self):
        # cart position, pole angle, cart speed, pole angular speed
        return np.array([self.data.qpos[0], self.data.qpos[1],
                         self.data.qvel[0], self.data.qvel[1]], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[0] = self.np_random.uniform(-0.05, 0.05)
        self.data.qpos[1] = self.np_random.uniform(-0.15, 0.15)
        self.data.qvel[:2] = self.np_random.uniform(-0.05, 0.05, size=2)
        mujoco.mj_forward(self.model, self.data)
        self._steps = 0
        return self._obs(), {}

    def step(self, action):
        value = float(np.clip(np.asarray(action).reshape(-1)[0], -1., 1.))
        force = 20.0 * value
        self.data.ctrl[0] = force
        for _ in range(5):
            mujoco.mj_step(self.model, self.data)
        self._steps += 1
        x, theta, vx, omega = self._obs()
        # Example dense reward, not the Mini Pupper reward function.
        reward = 1.0 - 2.0 * float(theta**2) - 0.05 * float(x**2) - 0.001 * force**2
        terminated = bool(abs(theta) > 0.7 or abs(x) > 2.3)
        truncated = bool(self._steps >= 500)
        return self._obs(), float(reward), terminated, truncated, {'force_newtons': force}

    def close(self):
        pass


## 4. Step MuJoCo once to inspect the environment

In [ ]:
from pendulum_env import PendulumEnv
env=PendulumEnv()
obs, info=env.reset(seed=42)
print("Observation:",obs, "shape",obs.shape)
print("Action space:",env.action_space)
print("Step:",env.step([0.0])[:3])

## 4A. Inside PPO: actor and critic network architecture

Our observation is `[x, θ, x_dot, θ_dot]` (shape `(4,)`), and the action is one normalized cart force (shape `(1,)`). The **actor** chooses actions using a Gaussian distribution; the **critic** predicts a scalar value; **PPO** optimizes both. We explicitly choose two 64-neuron hidden layers for each network.

Note: This notebook uses a custom MuJoCo cart-pole, not the built-in Gymnasium inverted pendulum.

In [ ]:
from stable_baselines3 import PPO
from torch import nn
import torch
from pendulum_env import PendulumEnv

inspect_env = PendulumEnv()
probe = PPO("MlpPolicy", inspect_env, seed=42, verbose=0,
            policy_kwargs={"net_arch": {"pi": [64, 64], "vf": [64, 64]},
                           "activation_fn": nn.Tanh})
obs, _ = inspect_env.reset(seed=42)
print("observations:", obs, "shape:", obs.shape)
print("actor hidden layers:", probe.policy.mlp_extractor.policy_net)
print("actor output:", probe.policy.action_net)
print("critic hidden layers:", probe.policy.mlp_extractor.value_net)
print("critic output:", probe.policy.value_net)
print("trainable parameters:", sum(p.numel() for p in probe.policy.parameters()))
with torch.no_grad():
    x = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    policy_dist = probe.policy.get_distribution(x).distribution
    critic_value = probe.policy.predict_values(x)
print("Gaussian mean shape:", tuple(policy_dist.mean.shape))
print("std shape:", tuple(policy_dist.stddev.shape))
print("critic V shape:", tuple(critic_value.shape))
inspect_env.close()


### What the network actually controls

The actor's sampled/mean output is one normalized action (not a force in newtons). `PendulumEnv.step()` clips it to `[-1,1]` and sets `data.ctrl[0] = 20 × action` N. The critic value output is used **only for learning**; it does not get sent to MuJoCo. The next cells train the network and inspect an action after learning.

## 5. Train PPO
This is a small learning experiment. If the policy is still weak, increase training steps.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
train_env=Monitor(PendulumEnv())
model=PPO("MlpPolicy",train_env,verbose=1,seed=42,n_steps=512,batch_size=64,
          policy_kwargs={"net_arch": {"pi": [64, 64], "vf": [64, 64]},
                         "activation_fn": nn.Tanh})
model.learn(total_timesteps=30000)
model.save("pendulum_ppo")
train_env.close()

### After training: inspect the learned actor distribution and value estimate

Compare with the untrained policy printed earlier. The network sizes are the same, but the learned weights now differ.

In [ ]:
import torch
obs, _ = env.reset(seed=101)
action, _ = model.predict(obs, deterministic=True)
with torch.no_grad():
    t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    mean = model.policy.get_distribution(t).distribution.mean
    value = model.policy.predict_values(t)
print("observation", obs)
print("trained mean", mean.numpy(), "action", action)
print("commanded MuJoCo force", 20 * float(action[0]), "N")
print("critic state value", value.numpy())


## 6. Evaluate a few episodes
Check whether the pole stays upright for longer than random control.

In [ ]:
import numpy as np
for episode in range(3):
    obs,_=env.reset(seed=episode+100)
    total=0.0
    for step in range(500):
        action,_=model.predict(obs,deterministic=True)
        obs,reward,terminated,truncated,info=env.step(action)
        total+=reward
        if terminated or truncated:break
    print(f"episode {episode+1}: steps={step+1}, return={total:.1f}")

## 7. Compare against a zero-force controller
Evaluate the same starting seeds with no control; longer episodes and larger returns are a sign of learning, but outcomes vary.

In [ ]:
for episode in range(3):
    obs,_=env.reset(seed=episode+100)
    total=0.0
    for step in range(500):
        obs,reward,terminated,truncated,info=env.step([0.0])
        total+=reward
        if terminated or truncated:break
    print(f"no force, episode {episode+1}: steps={step+1}, return={total:.1f}")
env.close()

## 8. Download your checkpoint
Keep a copy: Colab instances are temporary. The ROS 2 integration tutorial is designed for a ROS 2-capable Ubuntu machine, **not Colab**.

In [ ]:
from google.colab import files
files.download("pendulum_ppo.zip")

## Student exercise
Change the control-cost weight from `0.001` to `0.01` in the reward function, retrain with the same seed, and compare episode length.

Then proceed to the Mini Pupper advanced notebook once you understand state, action, reward, and PPO.